# YOLOv11 Training on Google Colab

This notebook allows you to train YOLOv11 models using Google Colab's free GPU resources.

## Features
- ✅ GPU acceleration (Tesla T4 or better)
- ✅ Easy data upload via Google Drive
- ✅ Comprehensive training with logging
- ✅ Real-time monitoring with TensorBoard
- ✅ Download trained models
- ✅ Validation and testing

## Workflow
1. Setup environment and check GPU
2. Clone repository or mount Google Drive
3. Upload/prepare your dataset
4. Configure training parameters
5. Train the model
6. Validate and test
7. Download results

---

## 1. Setup Environment

First, let's check if we have GPU access and install dependencies.

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages
!pip install -q ultralytics>=8.0.0
!pip install -q opencv-python opencv-contrib-python
!pip install -q matplotlib seaborn pandas
!pip install -q tensorboard

# Verify installation
import ultralytics
import torch
import cv2

print(f"✓ Ultralytics: {ultralytics.__version__}")
print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

## 2. Choose Your Method

Select one of the following methods:
- **Method A**: Clone from GitHub (recommended if you have the repo)
- **Method B**: Mount Google Drive (if you have data in Drive)
- **Method C**: Upload dataset directly

### Method A: Clone from GitHub

In [ ]:
# Clone your repository
!git clone https://github.com/xinghao2003/rt-object-tracking.git
%cd rt-object-tracking

# Create necessary directories
!mkdir -p data/train/images data/train/labels
!mkdir -p data/val/images data/val/labels
!mkdir -p data/test/images data/test/labels
!mkdir -p models logs reports

### Method B: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set working directory to your project in Drive
# Adjust the path to match your Google Drive structure
%cd /content/drive/MyDrive/rt-object-tracking

### Method C: Setup in Colab

In [ ]:
# Create project structure in Colab
!mkdir -p yolov11_training
%cd yolov11_training

# Create directory structure
!mkdir -p data/train/images data/train/labels
!mkdir -p data/val/images data/val/labels
!mkdir -p data/test/images data/test/labels
!mkdir -p models logs reports config

print("✓ Project structure created")
!tree -L 2 -d

## 3. Upload Your Dataset

### Option 1: Upload ZIP file containing your dataset

In [ ]:
from google.colab import files
import zipfile
import shutil

# Upload dataset ZIP file
print("Please upload your dataset ZIP file...")
uploaded = files.upload()

# Extract ZIP file
for filename in uploaded.keys():
    print(f"Extracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('data_temp')
    print(f"✓ Extracted {filename}")

# Check extracted structure
!ls -la data_temp/

### Option 2: Download from URL

In [ ]:
# Download dataset from URL (e.g., Roboflow, Kaggle)
# Example:
# !wget https://path-to-your-dataset.zip -O dataset.zip
# !unzip -q dataset.zip -d data_temp

# Or use Roboflow API
# !pip install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("workspace-name").project("project-name")
# dataset = project.version(1).download("yolov8")

### Split Dataset (if needed)

In [ ]:
import os
import random
import shutil
from pathlib import Path

def split_dataset(source_images, source_labels, output_dir, 
                  train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42):
    """Split dataset into train/val/test"""
    random.seed(seed)
    
    # Get image files
    image_files = list(Path(source_images).glob('*'))
    image_files = [f for f in image_files if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]
    
    print(f"Found {len(image_files)} images")
    
    # Shuffle
    random.shuffle(image_files)
    
    # Calculate splits
    n_total = len(image_files)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    
    train_files = image_files[:n_train]
    val_files = image_files[n_train:n_train + n_val]
    test_files = image_files[n_train + n_val:]
    
    print(f"Split: {len(train_files)} train, {len(val_files)} val, {len(test_files)} test")
    
    # Copy files
    for split_name, files in [('train', train_files), ('val', val_files), ('test', test_files)]:
        for img_file in files:
            # Copy image
            dst_img = Path(output_dir) / split_name / 'images' / img_file.name
            dst_img.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_file, dst_img)
            
            # Copy label
            label_file = Path(source_labels) / f"{img_file.stem}.txt"
            if label_file.exists():
                dst_label = Path(output_dir) / split_name / 'labels' / f"{img_file.stem}.txt"
                dst_label.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(label_file, dst_label)
    
    print("✓ Dataset split completed!")

# Example usage (uncomment and adjust paths):
# split_dataset(
#     source_images='data_temp/images',
#     source_labels='data_temp/labels',
#     output_dir='data',
#     train_ratio=0.7,
#     val_ratio=0.2,
#     test_ratio=0.1
# )

## 4. Create Dataset Configuration

In [ ]:
import yaml

# Configure your dataset
dataset_config = {
    'path': '/content/rt-object-tracking/data',  # Adjust based on your setup
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    
    # Update with your classes
    'nc': 3,  # Number of classes
    'names': {
        0: 'person',
        1: 'car',
        2: 'bicycle'
    }
}

# Save dataset YAML
with open('data/dataset.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False, sort_keys=False)

print("✓ Dataset YAML created")
print("\nDataset configuration:")
print(yaml.dump(dataset_config, default_flow_style=False))

### Validate Dataset

In [ ]:
from pathlib import Path

def validate_dataset(data_dir):
    """Validate dataset structure"""
    data_dir = Path(data_dir)
    
    print("=" * 80)
    print("DATASET VALIDATION")
    print("=" * 80)
    
    for split in ['train', 'val', 'test']:
        print(f"\n{split.upper()}:")
        img_dir = data_dir / split / 'images'
        lbl_dir = data_dir / split / 'labels'
        
        if not img_dir.exists():
            print(f"  ⚠ {img_dir} not found")
            continue
            
        images = list(img_dir.glob('*.[jp][pn]g')) + list(img_dir.glob('*.bmp'))
        labels = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
        
        print(f"  Images: {len(images)}")
        print(f"  Labels: {len(labels)}")
        
        # Count objects
        if lbl_dir.exists():
            total_objects = 0
            class_counts = {}
            
            for lbl_file in labels:
                with open(lbl_file, 'r') as f:
                    for line in f:
                        if line.strip():
                            total_objects += 1
                            class_id = int(line.split()[0])
                            class_counts[class_id] = class_counts.get(class_id, 0) + 1
            
            print(f"  Total objects: {total_objects}")
            for class_id, count in sorted(class_counts.items()):
                print(f"    Class {class_id}: {count} objects")
    
    print("\n" + "=" * 80)

# Validate your dataset
validate_dataset('data')

### Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from pathlib import Path

def visualize_samples(data_dir, split='train', num_samples=4):
    """Visualize sample images with labels"""
    img_dir = Path(data_dir) / split / 'images'
    lbl_dir = Path(data_dir) / split / 'labels'
    
    images = list(img_dir.glob('*.[jp][pn]g'))[:num_samples]
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 15))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(images[:4]):
        # Read image
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Read labels
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    if line.strip():
                        parts = line.strip().split()
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:5])
                        
                        # Convert YOLO format to bbox
                        x1 = int((x_center - width/2) * w)
                        y1 = int((y_center - height/2) * h)
                        x2 = int((x_center + width/2) * w)
                        y2 = int((y_center + height/2) * h)
                        
                        # Draw bbox
                        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                        cv2.putText(img, f"Class {class_id}", (x1, y1-10),
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
        
        axes[idx].imshow(img)
        axes[idx].set_title(img_path.name)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize samples
visualize_samples('data', 'train', num_samples=4)

## 5. Train YOLOv11 Model

Now let's train the model with your dataset.

In [ ]:
from ultralytics import YOLO
import torch

# Training configuration
CONFIG = {
    'model_size': 'n',        # n, s, m, l, x (n=fastest, x=most accurate)
    'task': 'detect',         # detect, segment, pose, classify
    'epochs': 100,            # Number of training epochs
    'imgsz': 640,             # Input image size
    'batch': 16,              # Batch size (reduce if out of memory)
    'data': 'data/dataset.yaml',  # Path to dataset YAML
    'project': 'models',      # Output directory
    'name': 'yolov11_custom', # Run name
    'device': 0,              # GPU device (0, 1, 2... or 'cpu')
    'patience': 50,           # Early stopping patience
    'save': True,             # Save checkpoints
    'save_period': 10,        # Save checkpoint every N epochs
    'cache': True,            # Cache images for faster training
    'workers': 8,             # Number of worker threads
}

print("Training Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

# Check GPU
print(f"\nUsing device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Initialize model
model_name = f"yolov11{CONFIG['model_size']}"
if CONFIG['task'] == 'segment':
    model_name += '-seg'
elif CONFIG['task'] == 'pose':
    model_name += '-pose'
elif CONFIG['task'] == 'classify':
    model_name += '-cls'
model_name += '.pt'

print(f"Initializing model: {model_name}")
model = YOLO(model_name)

# Start training
print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80 + "\n")

results = model.train(
    data=CONFIG['data'],
    epochs=CONFIG['epochs'],
    imgsz=CONFIG['imgsz'],
    batch=CONFIG['batch'],
    device=CONFIG['device'],
    project=CONFIG['project'],
    name=CONFIG['name'],
    patience=CONFIG['patience'],
    save=CONFIG['save'],
    save_period=CONFIG['save_period'],
    cache=CONFIG['cache'],
    workers=CONFIG['workers'],
    plots=True,
    verbose=True,
)

print("\n" + "="*80)
print("TRAINING COMPLETED")
print("="*80)

### Monitor Training with TensorBoard (Optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir models/yolov11_custom

### View Training Results

In [ ]:
from IPython.display import Image, display

# Display training curves
print("Training Curves:")
display(Image(filename=f'{CONFIG["project"]}/{CONFIG["name"]}/results.png'))

# Display confusion matrix
print("\nConfusion Matrix:")
display(Image(filename=f'{CONFIG["project"]}/{CONFIG["name"]}/confusion_matrix.png'))

# Display validation predictions
print("\nValidation Predictions:")
display(Image(filename=f'{CONFIG["project"]}/{CONFIG["name"]}/val_batch0_pred.jpg'))

## 6. Validate Model

In [ ]:
# Load best model
best_model_path = f"{CONFIG['project']}/{CONFIG['name']}/weights/best.pt"
model = YOLO(best_model_path)

# Run validation
print("Running validation on validation set...")
val_results = model.val(
    data=CONFIG['data'],
    split='val',
    imgsz=CONFIG['imgsz'],
    batch=CONFIG['batch'],
    device=CONFIG['device'],
    plots=True,
    verbose=True
)

# Print metrics
print("\n" + "="*80)
print("VALIDATION METRICS")
print("="*80)
print(f"mAP50: {val_results.box.map50:.4f}")
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"Precision: {val_results.box.p.mean():.4f}")
print(f"Recall: {val_results.box.r.mean():.4f}")
print("="*80)

## 7. Test Model

In [ ]:
# Run test
print("Running test on test set...")
test_results = model.val(
    data=CONFIG['data'],
    split='test',
    imgsz=CONFIG['imgsz'],
    batch=CONFIG['batch'],
    device=CONFIG['device'],
    plots=True,
    verbose=True
)

# Print metrics
print("\n" + "="*80)
print("TEST METRICS")
print("="*80)
print(f"mAP50: {test_results.box.map50:.4f}")
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"Precision: {test_results.box.p.mean():.4f}")
print(f"Recall: {test_results.box.r.mean():.4f}")
print("="*80)

## 8. Run Inference on Sample Images

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# Get sample images from test set
test_images = list(Path('data/test/images').glob('*.[jp][pn]g'))[:4]

# Run inference
results = model.predict(
    source=test_images,
    conf=0.25,
    iou=0.45,
    show_labels=True,
    show_conf=True
)

# Display results
fig, axes = plt.subplots(2, 2, figsize=(15, 15))
axes = axes.flatten()

for idx, (result, img_path) in enumerate(zip(results[:4], test_images[:4])):
    # Get annotated image
    annotated = result.plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    axes[idx].imshow(annotated)
    axes[idx].set_title(f"{img_path.name} - {len(result.boxes)} detections")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"Processed {len(results)} images")

## 9. Export Model (Optional)

Export to different formats for deployment.

In [ ]:
# Export to ONNX (for cross-platform deployment)
print("Exporting to ONNX...")
model.export(format='onnx', imgsz=CONFIG['imgsz'])
print("✓ ONNX export completed")

# Other export formats:
# model.export(format='torchscript')  # TorchScript
# model.export(format='tflite')       # TensorFlow Lite
# model.export(format='coreml')       # CoreML (for iOS)
# model.export(format='engine')       # TensorRT

## 10. Download Trained Model

Download your trained model to use locally.

In [ ]:
import shutil
from google.colab import files

# Create ZIP of results
print("Creating ZIP file of results...")
shutil.make_archive(
    'yolov11_trained_model',
    'zip',
    f"{CONFIG['project']}/{CONFIG['name']}"
)

print("\nDownloading trained model...")
files.download('yolov11_trained_model.zip')
print("✓ Download started")

In [ ]:
# Or download just the best model weights
print("Downloading best model weights...")
files.download(f"{CONFIG['project']}/{CONFIG['name']}/weights/best.pt")
print("✓ Download started")

### Save to Google Drive (Alternative)

In [ ]:
# Copy results to Google Drive
from google.colab import drive
import shutil

# Mount Drive if not already mounted
if not Path('/content/drive').exists():
    drive.mount('/content/drive')

# Copy to Drive
drive_path = '/content/drive/MyDrive/YOLOv11_Models'
!mkdir -p {drive_path}

print(f"Copying results to Google Drive: {drive_path}")
shutil.copytree(
    f"{CONFIG['project']}/{CONFIG['name']}",
    f"{drive_path}/{CONFIG['name']}",
    dirs_exist_ok=True
)
print("✓ Results saved to Google Drive")

## 11. Training Summary

In [ ]:
import json
from pathlib import Path

print("="*80)
print("TRAINING SUMMARY")
print("="*80)

print(f"\nModel: {model_name}")
print(f"Dataset: {CONFIG['data']}")
print(f"Epochs: {CONFIG['epochs']}")
print(f"Batch Size: {CONFIG['batch']}")
print(f"Image Size: {CONFIG['imgsz']}")

print("\nFinal Metrics:")
print(f"  Validation mAP50-95: {val_results.box.map:.4f}")
print(f"  Test mAP50-95: {test_results.box.map:.4f}")

print("\nOutput Files:")
print(f"  Best Model: {CONFIG['project']}/{CONFIG['name']}/weights/best.pt")
print(f"  Last Model: {CONFIG['project']}/{CONFIG['name']}/weights/last.pt")
print(f"  Results: {CONFIG['project']}/{CONFIG['name']}/results.png")

print("\n" + "="*80)
print("Training completed successfully!")
print("You can now download the model or save it to Google Drive.")
print("="*80)

## Next Steps

After training is complete:

1. **Download the trained model** using the cells above
2. **Use the model locally** with the GUI application:
   ```bash
   python src/gui/app.py
   ```
3. **Run inference from command line**:
   ```bash
   python src/inference/predict.py --model best.pt --source image.jpg
   ```
4. **Track objects in videos**:
   ```bash
   python src/inference/predict.py --model best.pt --source video.mp4 --task track
   ```

## Tips

- **Out of memory?** Reduce batch size: `CONFIG['batch'] = 8`
- **Training too slow?** Use smaller model: `CONFIG['model_size'] = 'n'`
- **Poor accuracy?** Try:
  - Increase epochs
  - Use larger model (s, m, l)
  - Increase dataset size
  - Check data quality
- **Want faster inference?** Export to ONNX or TensorRT

---

**Happy Training! 🚀**